# Turing patterns: Gray–Scott reaction–diffusion

*biophysics · advanced* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/biophysics/turing_patterns)

Two chemicals that react and diffuse at different speeds can turn a nearly uniform mixture into spots, stripes and mazes, Turing's 1952 idea for how animal coats and other biological patterns form. The Gray–Scott model is solved on a periodic grid with a 5-point Laplacian stencil for four $(F, k)$ pairs at once, in one $(2, B, n, n)$ tensor, and a linear-stability analysis (the dispersion relation $\lambda(q)$) shows where a genuine Turing instability exists.

**Vector form**

$$\partial_t u = D_u\nabla^2 u - u v^2 + F(1 - u),\qquad \partial_t v = D_v\nabla^2 v + u v^2 - (F + k)\,v,\qquad \lambda(q):\ \det\!\big(J - q^2\,\mathrm{diag}(D_u, D_v) - \lambda\,\mathbb 1\big) = 0$$

**Index (tensor) form**

$$(\nabla^2 u)_{bij} = \frac{u_{b,i+1,j} + u_{b,i-1,j} + u_{b,i,j+1} + u_{b,i,j-1} - 4u_{bij}}{h^2} \qquad u_{bij} \leftarrow u_{bij} + \Delta t\,\big[D_u(\nabla^2 u)_{bij} - u_{bij}v_{bij}^2 + F_b(1 - u_{bij})\big] \qquad M_{\alpha\beta}(q) = J_{\alpha\beta} - q^2 D_\alpha\,\delta_{\alpha\beta}$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $F_b,\ k_b$ | `parameters(device)` | (B, 1, 1) | feed and kill rate of pair $b$; the trailing 1s broadcast them over the grid |
| $u_{bij},\ v_{bij}$ | `y[0], y[1]` | (B, n, n) | substrate and autocatalyst concentration in cell $(i, j)$ for every pair, stacked into $y$ of shape $(2, B, n, n)$ |
| $(\nabla^2 u)_{bij}$ | `laplacian(u)` | (B, n, n) | four neighbours minus four times the centre, with torch.roll for the periodic edges |
| $u v^2,\ F(1-u),\ (F+k)v$ | `reaction(u, v, F, k)` | (B, n, n) | the autocatalytic step $U + 2V \to 3V$, the feed and the removal |
| $u \leftarrow u + \Delta t\,\dot u$ | `y + rhs(y, F, k, Du, Dv)` | (2, B, n, n) | explicit Euler with $\Delta t = 1$, stable while $D\,\Delta t / h^2 < 1/4$ |
| $(u^*, v^*)$ | `uniform_states(F, k)` | list | $(1, 0)$ always; two more where $F \ge 4(F + k)^2$ |
| $J_{\alpha\beta}$ | `jacobian(u, v, F, k)` | (2, 2) | linearised kinetics $\partial R_\alpha / \partial c_\beta$ at a uniform state |
| $\lambda(q)$ | `growth_rate(J, Du, Dv, q)` | (Q,) | largest $\operatorname{Re}$ eigenvalue of $M(q)$ for a batch of $Q$ wavenumbers |
| $\lambda_{\text{Turing}}$ | `turing_state(F, k, Du, Dv)` | (2,) | a state with $\lambda(0) < 0$ but $\lambda(q) > 0$ in a band: the Turing instability |

Turing's insight was that diffusion, which normally smooths everything out, can destabilise a uniform state when an autocatalyst (here $v$) spreads more slowly than the substance that limits it ($u$): a small bump of $v$ grows locally by eating $u$, and the depletion of $u$ around it stops neighbours from growing, a short-range activation and long-range inhibition. Linear stability makes this precise. A wave $e^{i\mathbf q\cdot\mathbf x + \lambda t}$ on a uniform state grows at the eigenvalues of $J - q^2\,\mathrm{diag}(D_u, D_v)$; if $\lambda(0) < 0$ but $\lambda(q) > 0$ for a band of $q$, the fastest wavelength $2\pi/q_{\max}$ appears out of tiny noise. With $D_u = D_v$ every eigenvalue just shifts by $-Dq^2$, so no Turing instability is possible: set the diffusion ratio to 1 and the blue region on the map disappears. In Gray–Scott only the "Turing" pair here is of that kind, and it starts from its uniform state plus 1 % noise; the dashed line checks that the amplitude first grows as $e^{\lambda_{\max} t}$. The famous spots, stripes and mazes (Pearson 1993) mostly live where the only uniform state is $u = 1$, $v = 0$, which is stable to every small perturbation; those patterns need a finite seed (a few squares of $v$) and are nonlinear, self-replicating structures rather than a linear Turing instability. The 5-point stencil sees a wave as $q^2 \to (4/h^2)\sin^2(qh/2)$, which is close to $q^2$ for wavelengths of many cells. Real biological examples include digit spacing in limbs, hair-follicle spacing and fish skin stripes, where the activator and inhibitor are signalling proteins.


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/biophysics/turing_patterns/main.py
"""Main loop: (F, k) pairs → linear stability (dispersion relation) → batched simulation → plot."""

import math
from pathlib import Path

import torch

from problems._common.runtime import cli, device, fast_mode

from . import plot, solve, tensors
from .equation import growth_rate, jacobian, uniform_states


def run(params: dict, out_dir: Path) -> Path:
    dev = device()
    fast = fast_mode()
    Du = tensors.DU
    Dv = params["ratio"] * Du
    n = 48 if fast else params["grid"]
    n_steps = 300 if fast else params["steps"]
    names = [name for name, _, _ in tensors.PAIRS]
    F, k = tensors.parameters(dev)  # (B, 1, 1) each
    print(f"Gray–Scott on a {n}×{n} periodic grid, D_u = {Du}, D_v = {Dv:.3f}, {len(names)} (F, k) pairs at once")

    # --- linear stability of the uniform states: the dispersion relation λ(q) ---
    q = torch.linspace(0, 1.2, 241, dtype=torch.float64)
    curves = {}  # name → list of (state, λ(q))
    for name, f, kk in tensors.PAIRS:
        curves[name] = []
        states = uniform_states(f, kk)
        for u, v in states[1:]:
            lam = growth_rate(jacobian(u, v, f, kk), Du, Dv, q)
            curves[name].append(((u, v), lam))
        print(f"{name:8s} F = {f:.4f}, k = {kk:.4f}: " + _verdict(states, curves[name], q))

    y0 = tensors.initial_state(n, Du, Dv, dev)  # (2, B, n, n)
    at, frames, amp = solve.solve(y0, F, k, Du, Dv, n_steps)
    wl = solve.dominant_wavelength(frames[-1])
    for b, name in enumerate(names):
        print(f"{name:8s} after {n_steps} steps: pattern wavelength ≈ {wl[b].item():.1f} cells")

    grid_F = torch.linspace(0.02, 0.09, 12 if fast else 71)
    grid_k = torch.linspace(0.054, 0.068, 12 if fast else 71)
    classes = solve.stability_map(grid_F, grid_k, Du, Dv)
    return plot.animate(
        at, frames, amp, tensors.PAIRS, curves, q, grid_F, grid_k, classes, out_dir / "turing_patterns.gif"
    )


def _verdict(states: list, curves: list, q: torch.Tensor) -> str:
    if len(states) == 1:
        return "only u = 1, v = 0 (stable for every q): patterns need a finite seed"
    parts = []
    for (u, v), lam in curves:
        i = int(lam.argmax())
        kind = "Turing-unstable" if lam[0] < 0 < lam[i] else ("unstable at q = 0" if lam[0] > 0 else "stable")
        text = f"(u, v) = ({u:.3f}, {v:.3f}) {kind}"
        if lam[0] < 0 < lam[i]:
            text += f", fastest q = {q[i]:.3f} (λ = {lam[i]:.4f}/step, wavelength {2 * math.pi / q[i]:.1f} cells)"
        parts.append(text)
    return "; ".join(parts)


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/biophysics/turing_patterns/equation.py
r"""Physics → tensor translation: Gray–Scott reaction–diffusion and its linear stability.

Indices: b parameter pair (F_b, k_b), (i, j) grid cell, q wavenumber.

    ∂u/∂t = D_u ∇²u - u v² + F (1 - u)          u: substrate ("food"), fed in at rate F
    ∂v/∂t = D_v ∇²v + u v² - (F + k) v          v: autocatalyst, removed at rate F + k

On the grid (spacing h, periodic) the Laplacian is the 5-point stencil
    (∇²u)_{bij} = (u_{b,i+1,j} + u_{b,i-1,j} + u_{b,i,j+1} + u_{b,i,j-1} - 4 u_{bij}) / h².
No index is summed in the update: every cell of every parameter pair advances on its own, coupled only to its
four neighbours by the stencil.

Linear stability of a uniform state (u*, v*): small waves δ ∝ exp(i q·x + λ t) grow at the eigenvalues of
    M(q) = J - q² diag(D_u, D_v),   J = [[-v*² - F, -2u*v*], [v*², 2u*v* - (F + k)]].
Turing instability: stable without diffusion (Re λ(0) < 0) but Re λ(q) > 0 for a band of q ≠ 0.
"""

import math

import torch


def laplacian(u: torch.Tensor, h: float = 1.0) -> torch.Tensor:
    """5-point periodic Laplacian of u (B, n, n): neighbour sum minus four times the centre, over h².   (B, n, n)"""
    neighbours = u.roll(1, -2) + u.roll(-1, -2) + u.roll(1, -1) + u.roll(-1, -1)
    return (neighbours - 4 * u) / h**2


def reaction(u: torch.Tensor, v: torch.Tensor, F: torch.Tensor, k: torch.Tensor):
    """Local kinetics R_u = -u v² + F (1 - u),  R_v = u v² - (F + k) v.   F, k: (B, 1, 1) broadcast over the grid."""
    uvv = u * v * v  # the autocatalytic step U + 2V → 3V
    return -uvv + F * (1 - u), uvv - (F + k) * v


def rhs(y: torch.Tensor, F: torch.Tensor, k: torch.Tensor, Du: float, Dv: float) -> torch.Tensor:
    """dy/dt for y = (u, v) of shape (2, B, n, n)."""
    u, v = y[0], y[1]
    Ru, Rv = reaction(u, v, F, k)
    return torch.stack([Du * laplacian(u) + Ru, Dv * laplacian(v) + Rv])


def uniform_states(F: float, k: float) -> list[tuple[float, float]]:
    """Uniform steady states (u*, v*): always (1, 0), and where F ≥ 4 (F + k)² two more,
    v* = [F ± √(F² - 4F(F + k)²)] / 2(F + k),  u* = (F + k) / v*   (from u v = F + k and F (1 - u) = u v²)."""
    states = [(1.0, 0.0)]
    disc = F * F - 4 * F * (F + k) ** 2
    if disc >= 0:
        for sign in (1.0, -1.0):
            v = (F + sign * math.sqrt(disc)) / (2 * (F + k))
            states.append(((F + k) / v, v))
    return states


def jacobian(u: float, v: float, F: float, k: float) -> torch.Tensor:
    """J = ∂(R_u, R_v)/∂(u, v) at a uniform state.   (2, 2)"""
    return torch.tensor([[-v * v - F, -2 * u * v], [v * v, 2 * u * v - (F + k)]], dtype=torch.float64)


def growth_rate(J: torch.Tensor, Du: float, Dv: float, q: torch.Tensor) -> torch.Tensor:
    """Dispersion relation: largest Re λ of M(q) = J - q² diag(D_u, D_v) for every wavenumber q (Q,).   (Q,)"""
    D = torch.diag(torch.tensor([Du, Dv], dtype=J.dtype))
    M = J - q.to(J.dtype)[:, None, None] ** 2 * D  # (Q, 2, 2)
    return torch.linalg.eigvals(M).real.max(-1).values


def lattice_q2(q: torch.Tensor, h: float = 1.0) -> torch.Tensor:
    """What the 5-point stencil does to a wave along one axis: -∇² exp(iqx) = (4/h²) sin²(qh/2) exp(iqx)."""
    return 4 / h**2 * torch.sin(q * h / 2) ** 2


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/biophysics/turing_patterns/tensors.py
"""Build the concrete tensors: the (F, k) parameter pairs and the starting concentrations on the grid.

Grid units: cell size h = 1 and time step 1, with D_u = 0.2 and by default D_v = D_u / 2, the values of
Pearson's 1993 classification of Gray–Scott patterns (D_u / h² ≈ 0.2 there too).
"""

import torch

from .equation import growth_rate, jacobian, uniform_states

DU = 0.2  # substrate diffusion, cells² per step

# name, feed rate F, kill rate k
PAIRS = [
    ("spots", 0.0367, 0.0649),
    ("stripes", 0.0545, 0.062),
    ("maze", 0.029, 0.057),
    ("Turing", 0.060, 0.062),
]


def parameters(device: torch.device) -> tuple[torch.Tensor, torch.Tensor]:
    """F_b and k_b, shape (B, 1, 1) each, so they broadcast over the (n, n) grid."""
    F = torch.tensor([f for _, f, _ in PAIRS], device=device)[:, None, None]
    k = torch.tensor([kk for _, _, kk in PAIRS], device=device)[:, None, None]
    return F, k


def turing_state(F: float, k: float, Du: float, Dv: float) -> tuple[float, float] | None:
    """The uniform state that is stable at q = 0 but unstable for some wavelength, if there is one."""
    q = torch.linspace(0, 1.5, 301)
    for u, v in uniform_states(F, k)[1:]:
        lam = growth_rate(jacobian(u, v, F, k), Du, Dv, q)
        if lam[0] < 0 < lam.max():
            return u, v
    return None


def initial_state(n: int, Du: float, Dv: float, device: torch.device, seed: int = 0) -> torch.Tensor:
    """y0 of shape (2, B, n, n).

    Where a Turing-unstable uniform state exists, start there with 1 % noise: the pattern has to grow out of
    nothing but the instability. Elsewhere the only stable uniform state is u = 1, v = 0, which small noise
    cannot destabilise, so a few square patches of v are dropped in to seed the pattern (a finite-amplitude start).
    """
    gen = torch.Generator().manual_seed(seed)
    u = torch.ones(len(PAIRS), n, n)
    v = torch.zeros(len(PAIRS), n, n)
    for b, (_, F, k) in enumerate(PAIRS):
        state = turing_state(F, k, Du, Dv)
        if state is not None:
            u[b], v[b] = state
            continue
        for _ in range(6):
            i, j = torch.randint(0, n - n // 12, (2,), generator=gen).tolist()
            u[b, i : i + n // 12, j : j + n // 12] = 0.5
            v[b, i : i + n // 12, j : j + n // 12] = 0.25
    y = torch.stack([u, v])
    y = y * (1 + 0.01 * torch.randn(y.shape, generator=gen))
    return y.to(device)


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/biophysics/turing_patterns/solve.py
"""Explicit Euler time stepping of the batched reaction–diffusion system, and the wavelength of a pattern."""

import torch

from problems._common.progress import steps

from .equation import growth_rate, jacobian, rhs, uniform_states


def solve(y0: torch.Tensor, F: torch.Tensor, k: torch.Tensor, Du: float, Dv: float, n_steps: int, n_frames: int = 90):
    """y ← y + dt · rhs(y) with dt = 1 (stable while D dt / h² < ¼).

    Returns (frame steps (T,), v snapshots (T, B, n, n), pattern amplitude std(v) after every step (n_steps + 1, B)).
    """
    every = max(1, n_steps // n_frames)
    y = y0.clone()
    at, frames, amp = [0], [y[1].clone()], [y[1].std((-1, -2))]
    for step in steps(n_steps, "reaction–diffusion steps"):
        y = y + rhs(y, F, k, Du, Dv)
        amp.append(y[1].std((-1, -2)))
        if (step + 1) % every == 0:
            at.append(step + 1)
            frames.append(y[1].clone())
    return torch.tensor(at), torch.stack(frames), torch.stack(amp)


def dominant_wavelength(v: torch.Tensor) -> torch.Tensor:
    """Wavelength (grid cells) of the strongest ring of the power spectrum |FFT v|² of each pattern v (B, n, n).

    Returns (B,)."""
    n = v.shape[-1]
    power = torch.fft.fft2(v - v.mean((-1, -2), keepdim=True)).abs() ** 2  # (B, n, n)
    f = torch.fft.fftfreq(n, device=v.device) * n  # waves per box along each axis
    rings = torch.sqrt(f[:, None] ** 2 + f[None, :] ** 2).round().long().clamp(max=n // 2)  # (n, n)
    spectrum = torch.zeros(v.shape[0], n // 2 + 1, device=v.device)
    spectrum.index_add_(1, rings.flatten(), power.flatten(1))  # sum the power on each ring
    spectrum[:, 0] = 0  # ignore the mean
    return n / spectrum.argmax(1).float()


def stability_map(F: torch.Tensor, k: torch.Tensor, Du: float, Dv: float) -> torch.Tensor:
    """Classify every (F, k) of a grid: 0 only u = 1, v = 0 exists; 1 more uniform states, none Turing-unstable;
    2 a uniform state that is stable at q = 0 but grows at some q ≠ 0 (Turing).   F (nF,), k (nk,) → (nF, nk)."""
    q = torch.linspace(0, 1.5, 151)
    out = torch.zeros(len(F), len(k), dtype=torch.long)
    for a, f in enumerate(F.tolist()):
        for b, kk in enumerate(k.tolist()):
            states = uniform_states(f, kk)[1:]
            if not states:
                continue
            out[a, b] = 1
            for u, v in states:
                lam = growth_rate(jacobian(u, v, f, kk), Du, Dv, q)
                if lam[0] < 0 < lam.max():
                    out[a, b] = 2
    return out


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/biophysics/turing_patterns/plot.py
"""Animate the patterns side by side, with the dispersion relation, the linear growth check and the (F, k) map."""

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from matplotlib.colors import ListedColormap

from problems._common.plotting import frame_indices, save_gif


def animate(
    at: torch.Tensor,
    frames: torch.Tensor,
    amp: torch.Tensor,
    pairs: list[tuple[str, float, float]],
    curves: dict,
    q: torch.Tensor,
    grid_F: torch.Tensor,
    grid_k: torch.Tensor,
    classes: torch.Tensor,
    path: Path,
) -> Path:
    """frames (T, B, n, n) of v at steps at (T,); amp (n_steps + 1, B); pairs = [(name, F, k), ...];
    curves[name] = [((u*, v*), λ(q)), ...] for every non-trivial uniform state."""
    names = [name for name, _, _ in pairs]
    steps_at, v = at.cpu().numpy(), frames.cpu().numpy()
    a = amp.cpu().numpy()
    qs = q.numpy()
    B = len(names)
    colours = [f"C{b}" for b in range(B)]

    fig = plt.figure(figsize=(10.5, 5.6))
    outer = fig.add_gridspec(2, 1, height_ratios=[1.1, 1], hspace=0.32, left=0.07, right=0.98, top=0.95, bottom=0.09)
    top = outer[0].subgridspec(1, B, wspace=0.08)
    bottom = outer[1].subgridspec(1, 3, wspace=0.42)
    ax_v = [fig.add_subplot(top[b]) for b in range(B)]
    ax_d, ax_g, ax_m = (fig.add_subplot(bottom[c]) for c in range(3))

    cmap = plt.get_cmap("magma", 10)  # a discrete colormap keeps the GIF small
    images = []
    for b in range(B):
        images.append(ax_v[b].imshow(v[0, b], cmap=cmap, vmin=0, vmax=0.45, interpolation="nearest"))
        ax_v[b].set_xticks([])
        ax_v[b].set_yticks([])
        ax_v[b].grid(False)
        for side in ax_v[b].spines.values():
            side.set_color(colours[b])
            side.set_linewidth(2)
    titles = [ax_v[b].set_title("", fontsize=9) for b in range(B)]

    # dispersion relation of every non-trivial uniform state
    turing = None
    for b, name in enumerate(names):
        for s, (_, lam) in enumerate(curves[name]):
            lam = lam.numpy()
            unstable = lam[0] < 0 < lam.max()
            ax_d.plot(qs, lam, color=colours[b], lw=2 if unstable else 1, ls="-" if s == 0 else ":")
            if unstable:
                turing = (b, lam)
                ax_d.fill_between(qs, 0, lam, where=lam > 0, color=colours[b], alpha=0.25)
    ax_d.axhline(0, color="#374151", lw=0.8)
    ax_d.set_ylim(-0.03, 0.03)
    ax_d.set_xlim(0, qs[-1])
    ax_d.set_xlabel("wavenumber q (1/cell)", labelpad=1)
    ax_d.tick_params(labelsize=7)
    ax_g.tick_params(labelsize=7)
    ax_d.set_ylabel("growth rate Re λ (1/step)", labelpad=1)
    ax_d.set_title("dispersion relation of uniform states", fontsize=9)
    ax_d.text(0.97, 0.94, "(1, 0): λ < 0 for all q", transform=ax_d.transAxes, ha="right", va="top", fontsize=7)

    # growth of the pattern amplitude: linear theory predicts exp(λ_max t) once the slower modes have died out
    t_all = np.arange(len(a))
    for b in range(B):
        ax_g.plot(t_all, a[:, b], color=colours[b], lw=1, alpha=0.35 if turing and b != turing[0] else 1)
    if turing is not None:
        b, lam = turing
        lam_max = lam.max()
        early = a[: len(a) // 2, b]
        t0 = int(early.argmin())
        t_line = np.linspace(t0, min(len(a) - 1, t0 + 6 / lam_max), 50)
        ax_g.plot(t_line, early[t0] * np.exp(lam_max * (t_line - t0)), "--", color="#374151", lw=1.2)
        label = f"dashed: exp(λ_max t)\nλ_max = {lam_max:.4f} / step"
        ax_g.text(0.04, 0.92, label, transform=ax_g.transAxes, va="top", fontsize=7)
    ax_g.set_yscale("log")
    ax_g.set_yticks([1e-3, 1e-2, 1e-1], ["0.001", "0.01", "0.1"])  # plain labels: no mathtext to typeset per frame
    ax_g.minorticks_off()
    ax_g.set_xlim(0, len(a) - 1)
    ax_g.set_xlabel("step", labelpad=1)
    ax_g.set_ylabel("amplitude std(v)", labelpad=1)
    ax_g.set_title("growth from a uniform state", fontsize=9)
    (marker,) = ax_g.plot([], [], "o", color="#111827", ms=4)

    # where in the (k, F) plane Turing patterns are possible
    shades = ListedColormap(["#ffffff", "#e5e7eb", "#93c5fd"])
    kk, ff = grid_k.numpy(), grid_F.numpy()
    ax_m.imshow(
        classes.numpy(),
        cmap=shades,
        vmin=0,
        vmax=2,
        origin="lower",
        aspect="auto",
        extent=(kk[0], kk[-1], ff[0], ff[-1]),
        interpolation="nearest",
    )
    k_line = np.linspace(kk[0], kk[-1], 200)
    F_line = (1 - 8 * k_line - np.sqrt(np.clip(1 - 16 * k_line, 0, None))) / 8  # F = 4 (F + k)², lower root
    ax_m.plot(k_line, F_line, color="#9ca3af", lw=1)
    ax_m.plot(k_line, (1 - 8 * k_line + np.sqrt(np.clip(1 - 16 * k_line, 0, None))) / 8, color="#9ca3af", lw=1)
    for b, (_, f, k) in enumerate(pairs):
        ax_m.plot(k, f, "o", color=colours[b], ms=6, mec="#111827")
    ax_m.set_xlim(kk[0], kk[-1])
    ax_m.set_ylim(ff[0], ff[-1])
    ax_m.set_xlabel("kill rate k", labelpad=1)
    ax_m.set_ylabel("feed rate F", labelpad=1)
    ax_m.set_title("blue: Turing-unstable, grey: other states", fontsize=9)
    ax_m.tick_params(labelsize=7)

    def update(t: int):
        for b in range(B):
            images[b].set_data(v[t, b])
            titles[b].set_text(f"{names[b]}, step {steps_at[t]}")
        i = steps_at[t]
        marker.set_data([i], [a[i, turing[0] if turing else 0]])

    return save_gif(fig, update, frame_indices(len(steps_at), 60), path, fps=8)


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'grid': 128,
    'steps': 10000,
    'ratio': 0.5
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.biophysics.turing_patterns")]:
    del sys.modules[name]
main = importlib.import_module("problems.biophysics.turing_patterns.main")
problem_dir = Path("problems/biophysics/turing_patterns")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
